In [1]:
from cats_ai.evaluation import experiment

In [2]:

if __name__ == "__main__":
    experiment()
    #query("CarCrash/videos/Normal/000007.mp4", ACCIDENT_PREDICTION_PROMPT, True)

Temp folder: /tmp/carcrash_masked_26w_78i8


KeyError: 'HF_API_KEY'

In [11]:
from cats_ai.prompts import ACCIDENT_ANALYSIS_PROMPT, ACCIDENT_ANALYSIS_SCHEMA
from cats_ai.config import MAX_NEW_TOKENS, NFRAMES
import cv2
from cats_ai.validation import validate_json
import base64
from openai import OpenAI
import os


def capture_frames(video_path: str, n_frames: int):
    cap = cv2.VideoCapture(video_path)
    base64Frames = []

    for i in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break

        _, buffer = cv2.imencode(".jpg", frame)
        encoded_image = base64.b64encode(buffer).decode("utf-8")
        base64Frames.append(encoded_image)

    cap.release()
    return base64Frames


def chatgpt_query(video_path: str, prompt: str, n_frames: int) -> str:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    base64_frames = capture_frames(video_path, n_frames)

    content = [
        {"type": "input_text", "text": prompt},
        *[
            {
                "type": "input_image",
                "image_url": f"data:image/jpeg;base64,{frame}",
            }
            for frame in base64_frames
        ],
    ]

    response = client.responses.create(
        model="gpt-5.5",
        input=[{"role": "user", "content": content}],
        max_output_tokens=650,
    )

    return response.output_text


out = chatgpt_query(
    "CarCrash/videos/Crash-1500/000001.mp4", ACCIDENT_ANALYSIS_PROMPT, NFRAMES
)


In [12]:
from cats_ai.validation import validate_json
from cats_ai.prompts import ACCIDENT_ANALYSIS_SCHEMA
validate_json(out, ACCIDENT_ANALYSIS_SCHEMA)


{'accident_present': True,
 'confidence': 0.96,
 'scene_description': 'The dashcam vehicle is traveling on a city street toward a signalized intersection with other vehicles nearby. A dark car enters from the right side of the intersection and crosses directly in front of the dashcam vehicle, making contact with the front of the dashcam car.',
 'main_objects': ['dashcam vehicle',
  'dark sedan/wagon crossing from the right',
  'light-colored car ahead near the right curb',
  'oncoming vehicles',
  'traffic lights',
  'intersection',
  'roadside trees and buildings'],
 'risk_factors': ['crossing paths at the intersection',
  'insufficient gap between the dark car and dashcam vehicle',
  'limited reaction distance',
  'close proximity of vehicles near the intersection'],
 'reasoning': "A clear impact is visible when the dark car moves across the dashcam vehicle's path and its side/front area contacts the dashcam vehicle's hood area. The vehicles remain in contact or extremely close immed